In [1]:
!pip install azure-ai-projects azure-core

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.3/274.3 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 4.1 MB/s eta 0:00:00


In [2]:
pip install --upgrade openai azure-ai-projects azure-identity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.0 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.31.0
    Uninstalling openai-2.31.0:
      Successfully uninstalled openai-2.31.0


In [3]:
from openai import OpenAI
import os
from google.colab import userdata

# Retrieve the secret key from Google Colab
api_key = userdata.get("azure-key")

client = OpenAI(
    api_key=api_key,  # Picks up the secret key from Colab environment
    base_url="https://rg-aai-demo-resource.openai.azure.com/openai/v1"
)

In [10]:
# 1. Define the input data
medical_report_data = """
Patient Name: John Smith
Age: 45
Gender: Male

Test Results:
Hemoglobin: 11.2 g/dL
WBC Count: 14500 /uL
Blood Sugar (Fasting): 132 mg/dL
Cholesterol: 245 mg/dL
Vitamin D: 18 ng/mL

Doctor Notes:
Patient reports fatigue, mild fever, and weakness.
"""

# 2. Extract structured medical report data
try:
    response = client.responses.create(
        model="gpt-4.1",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": """Extract the following details from the medical report and return ONLY valid JSON:

                        - Patient Name
                        - Age
                        - Gender
                        - Summary
                        - Abnormal Values
                        - Possible Conditions
                        - Recommendations
                        """
                    },
                    {
                        "type": "input_text",
                        "text": f"MEDICAL REPORT:\n{medical_report_data}"
                    }
                ]
            }
        ]
    )

    print("--- EXTRACTED MEDICAL REPORT DATA ---")
    print(response.output[0].content)

except Exception as e:
    print(f"Extraction Error: {e}")

--- EXTRACTED MEDICAL REPORT DATA ---
[ResponseOutputText(annotations=[], text='{\n  "Patient Name": "John Smith",\n  "Age": 45,\n  "Gender": "Male",\n  "Summary": "The patient reports fatigue, mild fever, and weakness. Blood tests show low hemoglobin, elevated WBC count, high fasting blood sugar, high cholesterol, and low vitamin D levels.",\n  "Abnormal Values": {\n    "Hemoglobin": "11.2 g/dL (Low)",\n    "WBC Count": "14500 /uL (High)",\n    "Blood Sugar (Fasting)": "132 mg/dL (High)",\n    "Cholesterol": "245 mg/dL (High)",\n    "Vitamin D": "18 ng/mL (Low)"\n  },\n  "Possible Conditions": [\n    "Anemia",\n    "Infection or inflammation",\n    "Pre-diabetes or diabetes",\n    "Hypercholesterolemia",\n    "Vitamin D deficiency"\n  ],\n  "Recommendations": [\n    "Further investigation for source of infection or inflammation",\n    "Evaluate and manage anemia",\n    "Lifestyle modification and monitoring for blood sugar and cholesterol",\n    "Vitamin D supplementation",\n    "Foll

In [11]:
import json

try:
    # 1. Get the first item in the content list
    # 2. Access the text returned from model response
    output_text_item = response.output[0].content[0]

    # Supports both dict and object formats
    if isinstance(output_text_item, dict):
        raw_json_string = output_text_item.get("text", "")
    else:
        raw_json_string = output_text_item.text

    # Parse JSON string
    extracted_data = json.loads(raw_json_string)

    print("--- SUCCESS: RESEARCH PAPER SUMMARY PARSED ---")
    print(json.dumps(extracted_data, indent=2))

except Exception as e:
    print(f"Parsing Error: {e}")

    # Troubleshooting
    try:
        print(f"Actual content type: {type(response.output[0].content[0])}")
    except:
        print("Unable to inspect response format.")

--- SUCCESS: RESEARCH PAPER SUMMARY PARSED ---
{
  "Patient Name": "John Smith",
  "Age": 45,
  "Gender": "Male",
  "Summary": "The patient reports fatigue, mild fever, and weakness. Blood tests show low hemoglobin, elevated WBC count, high fasting blood sugar, high cholesterol, and low vitamin D levels.",
  "Abnormal Values": {
    "Hemoglobin": "11.2 g/dL (Low)",
    "WBC Count": "14500 /uL (High)",
    "Blood Sugar (Fasting)": "132 mg/dL (High)",
    "Cholesterol": "245 mg/dL (High)",
    "Vitamin D": "18 ng/mL (Low)"
  },
  "Possible Conditions": [
    "Anemia",
    "Infection or inflammation",
    "Pre-diabetes or diabetes",
    "Hypercholesterolemia",
    "Vitamin D deficiency"
  ],
  "Recommendations": [
    "Further investigation for source of infection or inflammation",
    "Evaluate and manage anemia",
    "Lifestyle modification and monitoring for blood sugar and cholesterol",
    "Vitamin D supplementation",
    "Follow-up visit and possible referral to a specialist"
  ]
}


In [6]:
from azure.storage.blob import BlobServiceClient

# ✅ Correct connection string
AZURE_STORAGE_CONNECTION_STRING = "DefaultEndpointsProtocol=https;AccountName=scmdobuildblob;AccountKey=67eOvlgX0V2v1MnLNOdPTJL0IhSLldg3HF3VV9ISZyo61L50eRGF7z/7tlsqpgvfQBKqM2hm09rc+ASt4xJhOQ==;EndpointSuffix=core.windows.net"

blob_service_client = BlobServiceClient.from_connection_string(
    AZURE_STORAGE_CONNECTION_STRING
)

container_name = "blobstoragecontainer"

blob_client = blob_service_client.get_blob_client(
    container=container_name,
    blob="result.txt"
)

blob_client.upload_blob("Test upload", overwrite=True)

print("✅ Upload successful")

✅ Upload successful


In [12]:
import datetime

file_name = f"medicalreport_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

In [13]:
blob_client = blob_service_client.get_blob_client(
    container="blobstoragecontainer",
    blob=file_name
)

blob_client.upload_blob(response.output_text, overwrite=True)

{'etag': '"0x8DE9F85B048B3B5"',
 'last_modified': datetime.datetime(2026, 4, 21, 9, 9, 27, tzinfo=datetime.timezone.utc),
 'content_md5': bytearray(b'R\xb6.\xa9\xb7\x0fE\x9c\x81\xf5\xdc\xd0\xbe4\x95\x9b'),
 'client_request_id': 'cb93b298-3d61-11f1-97a0-0242ac1c000c',
 'request_id': '0aaa6928-101e-000e-656e-d192a4000000',
 'version': '2026-02-06',
 'version_id': None,
 'date': datetime.datetime(2026, 4, 21, 9, 9, 26, tzinfo=datetime.timezone.utc),
 'request_server_encrypted': True,
 'encryption_key_sha256': None,
 'encryption_scope': None,
 'structured_body': None}

In [14]:
print(f"✅ Stored as: {file_name}")

✅ Stored as: medicalreport_20260421_090924.txt
